# Media Cloud — National Collections, High-Volume Sources

This notebook:
1. Fetches all collections whose name contains **`- National`**
2. Pulls every source from those collections
3. Filters to sources with **content per week > 300**
4. Exports the result to CSV

**Prerequisites:**  
- Free Media Cloud account → https://search.mediacloud.org/sign-up  
- Your API key (profile page after login)  
- Rate limit: 4,000 API calls/week; some endpoints capped at 2 req/min

---

In [1]:
!pip install mediacloud --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import mediacloud.api
import pandas as pd
import time
import json
import os
from pathlib import Path
from datetime import datetime
from importlib.metadata import version

print(f"mediacloud {version('mediacloud')}")
print(f"Started: {datetime.now().isoformat()}")

mediacloud 5.0.0
Started: 2026-04-24T16:35:09.353502


## 1 — API key & config

In [3]:
# ── Set your API key (pick one method) ──
# MC_API_KEY = os.environ.get("MC_API_KEY", "")
MC_API_KEY = "2c8f2e2c530f3a4e957eb517e28bb98f0f9edef8"

if not MC_API_KEY:
    raise ValueError(
        "No API key found.\n"
        "Set MC_API_KEY env var or paste it above.\n"
        "Sign up free: https://search.mediacloud.org/sign-up"
    )

mc = mediacloud.api.DirectoryApi(MC_API_KEY)
print(f"API key loaded (ends ...{MC_API_KEY[-4:]})  ✓")

API key loaded (ends ...def8)  ✓


In [22]:
# ── Configuration ──
COLLECTION_NAME_FILTER = "- National"   # substring to match in collection names
MIN_CONTENT_PER_WEEK = 900              # minimum stories/week threshold
PAGE_SIZE = 100                         # results per API page
SLEEP_BETWEEN_PAGES = 0.5               # seconds between API calls
SLEEP_BETWEEN_COLLECTIONS = 1.0         # seconds between collections
CHECKPOINT_FILE = "_national_checkpoint.json"

print(f"Will fetch collections matching: '{COLLECTION_NAME_FILTER}'")
print(f"Will keep sources with content/week > {MIN_CONTENT_PER_WEEK}")

Will fetch collections matching: '- National'
Will keep sources with content/week > 900


## 2 — Find all "- National" collections

In [5]:
def fetch_all_collections(mc, page_size=PAGE_SIZE, sleep=SLEEP_BETWEEN_PAGES):
    """Page through all collections in the directory."""
    collections = []
    offset = 0
    page = 0
    while True:
        page += 1
        try:
            resp = mc.collection_list(limit=page_size, offset=offset)
        except Exception as e:
            print(f"  ⚠ Error page {page}: {e} — retrying in 60s")
            time.sleep(60)
            continue
        batch = resp.get("results", [])
        collections.extend(batch)
        if page % 5 == 0 or resp.get("next") is None:
            print(f"  Page {page}: +{len(batch)}  (total: {len(collections)})")
        if resp.get("next") is None or len(batch) == 0:
            break
        offset += len(batch)
        time.sleep(sleep)
    return collections


print("Fetching full list of collections...")
all_collections = fetch_all_collections(mc)
print(f"\nTotal collections in Media Cloud: {len(all_collections)}")

Fetching full list of collections...
  Page 5: +100  (total: 500)
  Page 10: +100  (total: 1000)
  Page 15: +100  (total: 1500)
  Page 18: +61  (total: 1761)

Total collections in Media Cloud: 1761


In [6]:
# Filter to collections whose name contains '- National'
national_collections = [
    c for c in all_collections
    if COLLECTION_NAME_FILTER.lower() in c.get("name", "").lower()
]

df_national_colls = pd.DataFrame(national_collections)
print(f"Collections matching '{COLLECTION_NAME_FILTER}': {len(national_collections)}\n")

display_cols = [col for col in ["id", "name", "source_count", "description"]
                if col in df_national_colls.columns]
df_national_colls[display_cols]

Collections matching '- National': 243



,id,name,source_count
0,34412043,Argentina - National,1895
1,34412427,Mexico - National,647
2,38376341,Nigeria - National,440
3,34412056,Japan - National,427
4,34412356,Spain - National,376
...,...,...,...
238,34412040,Svalbard and Jan Mayen - National,1
239,34412261,Pitcairn - National,1
240,34412425,Montserrat - National,1
241,34412290,Tuvalu - National,0


In [7]:
# Save the list of national collections for reference
df_national_colls.to_csv("national_collections.csv", index=False)
print(f"Saved {len(df_national_colls)} national collections → national_collections.csv")

Saved 243 national collections → national_collections.csv


## 3 — Retrieve all sources from those collections

In [8]:
def fetch_sources_for_collection(mc, coll_id, page_size=PAGE_SIZE, sleep=SLEEP_BETWEEN_PAGES):
    """Paginate through all sources in one collection."""
    sources = []
    offset = 0
    while True:
        try:
            resp = mc.source_list(collection_id=coll_id, limit=page_size, offset=offset)
        except Exception as e:
            print(f"    ⚠ Error offset={offset}: {e} — retrying in 60s")
            time.sleep(60)
            continue
        batch = resp.get("results", [])
        sources.extend(batch)
        if resp.get("next") is None or len(batch) == 0:
            break
        offset += len(batch)
        time.sleep(sleep)
    return sources

In [11]:
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

CHECKPOINT_FILE = "checkpoint.json"
max_workers = 5  # tune this (start small)

# ── Shared state ──
all_sources = {}           # source_id → source dict
source_to_collections = {} # source_id → list of collection names
done_ids = set()

lock = Lock()
t0 = time.time()

# ── Resume from checkpoint ──
if Path(CHECKPOINT_FILE).exists():
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    all_sources = {int(k): v for k, v in ckpt.get("sources", {}).items()}
    source_to_collections = {int(k): v for k, v in ckpt.get("s2c", {}).items()}
    done_ids = set(ckpt.get("done_ids", []))

    print(f"Resumed: {len(all_sources)} sources, {len(done_ids)} collections done")

total = len(national_collections)

# ── Worker ──
def process_collection(coll):
    coll_id = coll["id"]
    coll_name = coll.get("name", "")
    try:
        sources = fetch_sources_for_collection(mc, coll_id)
    except Exception as e:
        print(f"❌ Error in {coll_name}: {e}")
        sources = []
    return coll_id, coll_name, sources

# ── Submit tasks ──
futures = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    for coll in national_collections:
        if coll["id"] not in done_ids:
            futures.append(executor.submit(process_collection, coll))

    completed = len(done_ids)

    # ── Process results as they complete ──
    for future in as_completed(futures):
        coll_id, coll_name, sources = future.result()

        with lock:
            for s in sources:
                sid = s.get("id")
                if sid not in all_sources:
                    all_sources[sid] = s
                    source_to_collections[sid] = []
                source_to_collections[sid].append(coll_name)

            done_ids.add(coll_id)
            completed += 1

        elapsed = time.time() - t0
        print(f"[{completed}/{total}] {coll_name} → {len(sources)} sources "
              f"| Unique total: {len(all_sources)} "
              f"| {elapsed/60:.1f} min")

        # ── Checkpoint every 20 completed collections ──
        if completed % 20 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump({
                    "sources": {str(k): v for k, v in all_sources.items()},
                    "s2c": {str(k): v for k, v in source_to_collections.items()},
                    "done_ids": list(done_ids)
                }, f)
            print("💾 Checkpoint saved")

print(f"\n✅ Done! {len(all_sources)} unique sources from {total} collections.")

[1/243] Spain - National → 376 sources | Unique total: 376 | 0.1 min
[2/243] Japan - National → 427 sources | Unique total: 788 | 0.1 min
[3/243] Nigeria - National → 440 sources | Unique total: 1169 | 0.1 min
[4/243] South Africa - National → 344 sources | Unique total: 1500 | 0.1 min
[5/243] Mexico - National → 647 sources | Unique total: 2072 | 0.1 min
[6/243] Bangladesh - National → 257 sources | Unique total: 2299 | 0.1 min
[7/243] France - National → 343 sources | Unique total: 2642 | 0.2 min
[8/243] Russian Federation - National → 243 sources | Unique total: 2871 | 0.2 min
[9/243] Colombia - National → 244 sources | Unique total: 3112 | 0.2 min
[10/243] United States - National → 248 sources | Unique total: 3357 | 0.2 min
[11/243] Norway - National → 238 sources | Unique total: 3590 | 0.2 min
[12/243] Sweden - National → 214 sources | Unique total: 3804 | 0.2 min
[13/243] China - National → 234 sources | Unique total: 3997 | 0.2 min
[14/243] Ukraine - National → 235 sources | Un

## 4 — Discover the "content per week" field name

Media Cloud documents this as "the average number of stories published per week",
but the exact JSON key varies across API versions. We inspect the first record to find it.

In [12]:
# Inspect one source record to see all available fields
sample = next(iter(all_sources.values()))
print("All fields in a source record:\n")
for k, v in sorted(sample.items()):
    print(f"  {k:30s} = {repr(v)[:80]}")

All fields in a source record:

  alternative_domains            = []
  collection_count               = 8
  created_at                     = '2022-12-23T17:43:28.547804Z'
  homepage                       = 'http://europapress.es/'
  id                             = 41209
  label                          = 'europapress.es'
  last_rescraped                 = '2026-02-11T16:26:13.948948Z'
  last_rescraped_msg             = '0/0 added, 0/2 confirmed'
  last_story                     = '04/2026'
  media_type                     = None
  modified_at                    = '2026-04-18T09:42:16.286805Z'
  monitored                      = True
  name                           = 'europapress.es'
  notes                          = None
  platform                       = 'online_news'
  primary_language               = 'es'
  pub_country                    = None
  pub_state                      = None
  stories_date_empty             = 12254
  stories_date_future            = 0
  stories_date_past

In [13]:
# ── Auto-detect the stories-per-week field ──
# Known candidate names across API versions:
CANDIDATE_FIELDS = [
    "stories_per_week",
    "content_per_week",
    "story_count_per_week",
    "avg_stories_per_week",
    "num_stories_90",
    "stories_per_day",
]

cpw_field = None
for candidate in CANDIDATE_FIELDS:
    if candidate in sample:
        cpw_field = candidate
        break

# Fallback: look for any field with 'stor' or 'content' AND 'week' or 'per'
if cpw_field is None:
    for k in sample.keys():
        kl = k.lower()
        if ("stor" in kl or "content" in kl) and ("week" in kl or "per" in kl):
            cpw_field = k
            break

if cpw_field:
    print(f"✅ Using field: '{cpw_field}'")
    print(f"   Sample value: {sample[cpw_field]}")
else:
    print("⚠ Could not auto-detect the content-per-week field.")
    print("  Check the field list above and set cpw_field manually:")
    print('  cpw_field = "your_field_name_here"')

✅ Using field: 'stories_per_week'
   Sample value: 4356


## 5 — Build DataFrame, add collections column, filter

In [14]:
df_all = pd.DataFrame(all_sources.values())

# Add a column with the list of national collections each source belongs to
df_all["national_collections"] = df_all["id"].map(
    lambda sid: " | ".join(sorted(set(source_to_collections.get(sid, []))))
)

print(f"All sources (before filter): {len(df_all)}")
print(f"Columns: {list(df_all.columns)}")
df_all.head(3)

All sources (before filter): 14433
Columns: ['id', 'name', 'url_search_string', 'label', 'homepage', 'notes', 'platform', 'stories_per_week', 'last_story', 'created_at', 'modified_at', 'pub_country', 'pub_state', 'primary_language', 'media_type', 'last_rescraped', 'last_rescraped_msg', 'collection_count', 'alternative_domains', 'stories_total', 'stories_date_past', 'stories_date_future', 'stories_date_empty', 'monitored', 'national_collections']


,id,name,url_search_string,label,homepage,notes,platform,stories_per_week,last_story,created_at,...,last_rescraped,last_rescraped_msg,collection_count,alternative_domains,stories_total,stories_date_past,stories_date_future,stories_date_empty,monitored,national_collections
0,41209,europapress.es,None,europapress.es,http://europapress.es/,None,online_news,4356.0,04/2026,2022-12-23T17:43:28.547804Z,...,2026-02-11T16:26:13.948948Z,"0/0 added, 0/2 confirmed",8,[],1006624.0,17.0,0.0,12254.0,True,Spain - National
1,179018,larazon.es,None,larazon.es,http://larazon.es/,None,online_news,3923.0,04/2026,2022-12-23T17:43:28.547804Z,...,2026-02-22T15:54:12.506892Z,"2/2 added, 0/9 confirmed",7,[],289436.0,0.0,0.0,37694.0,True,Spain - National
2,41735,cadenaser.com,None,cadenaser,http://www.cadenaser.com/,None,online_news,3836.0,04/2026,2022-12-23T17:43:28.547804Z,...,2026-02-12T11:29:35.440130Z,"1/1 added, 0/38 confirmed",7,[],299276.0,14.0,0.0,210.0,True,Spain - National


In [23]:
# ── Filter: content per week > 900 ──

if cpw_field and cpw_field in df_all.columns:
    # Coerce to numeric (some may be None/NaN)
    df_all[cpw_field] = pd.to_numeric(df_all[cpw_field], errors="coerce")

    df_filtered = df_all[df_all[cpw_field] > MIN_CONTENT_PER_WEEK].copy()
    df_filtered = df_filtered.sort_values(cpw_field, ascending=False)

    print(f"Sources with {cpw_field} > {MIN_CONTENT_PER_WEEK}: {len(df_filtered)}")
    print(f"  (dropped {len(df_all) - len(df_filtered)} sources below threshold)")
else:
    print(f"⚠ Field '{cpw_field}' not found in DataFrame.")
    print("  Saving unfiltered. You can filter manually after inspecting the columns.")
    df_filtered = df_all.copy()

Sources with stories_per_week > 900: 488
  (dropped 13945 sources below threshold)


In [24]:
# Quick preview of the filtered data
preview_cols = [c for c in ["id", "name", "domain", "homepage", "url",
                            cpw_field, "pub_country", "primary_language",
                            "media_type", "national_collections"]
                if c and c in df_filtered.columns]

df_filtered[preview_cols].head(15)

,id,name,homepage,stories_per_week,pub_country,primary_language,media_type,national_collections
5583,219540,bhaskar.com,https://www.bhaskar.com/,45792.0,IND,hi,print_native,India - National
5122,879114,business24.ro,http://business24.ro/,27348.0,None,ro,None,Romania - National
11615,180285,news.de,http://news.de/,26208.0,DEU,de,digital_native,Germany - National (Media Impact Monitor Test)...
5584,262319,amarujala.com,https://www.amarujala.com/,23140.0,IND,hi,print_native,India - National
5982,657923,fnnews.com,http://www.fnnews.com/,18852.0,KOR,ko,print_native,"Korea, Republic of - National"
2871,126565,infobae.com,http://infobae.com/,15974.0,ARG,es,digital_native,Argentina - National | Colombia - National
5585,115065,indiatimes.com,http://indiatimes.com/,14455.0,IND,en,digital_native,India - National
5586,39784,indiatimes.com,http://timesofindia.indiatimes.com/,10213.0,IND,en,print_native,India - National
376,376850,prtimes.jp,http://prtimes.jp/,9566.0,None,ja,None,Japan - National
2072,932549,bbc.com,http://bbc.com,9037.0,GBR,en,None,Afghanistan - National | Bangladesh - National...


## 6 — Export to CSV

In [25]:
ts = datetime.now().strftime("%Y%m%d")

# Filtered high-volume sources
filtered_file = f"national_sources_cpw_gt_{MIN_CONTENT_PER_WEEK}_{ts}.csv"
df_filtered.to_csv(filtered_file, index=False)
print(f"✅ {len(df_filtered)} high-volume sources → {filtered_file}")

# Also save the full unfiltered set for reference
all_file = f"national_sources_all_{ts}.csv"
df_all.to_csv(all_file, index=False)
print(f"✅ {len(df_all)} total sources (unfiltered) → {all_file}")

✅ 488 high-volume sources → national_sources_cpw_gt_900_20260424.csv
✅ 14433 total sources (unfiltered) → national_sources_all_20260424.csv


## 7 — Summary stats

In [18]:
print("=" * 60)
print(f"National collections found:      {len(national_collections)}")
print(f"Total unique sources (all):      {len(df_all):,}")
print(f"Sources with cpw > {MIN_CONTENT_PER_WEEK}:          {len(df_filtered):,}")
if cpw_field and cpw_field in df_filtered.columns:
    print(f"Max {cpw_field}:          {df_filtered[cpw_field].max():,.0f}")
    print(f"Median {cpw_field}:       {df_filtered[cpw_field].median():,.0f}")
    print(f"Mean {cpw_field}:         {df_filtered[cpw_field].mean():,.0f}")
print("=" * 60)

National collections found:      243
Total unique sources (all):      14,433
Sources with cpw > 1000:          436
Max stories_per_week:          45,792
Median stories_per_week:       1,579
Mean stories_per_week:         2,375


In [19]:
# Country distribution in the filtered set
country_col = next(
    (c for c in ["pub_country", "country"] if c in df_filtered.columns), None
)
if country_col:
    print(f"Top 25 countries in filtered sources ('{country_col}'):\n")
    print(df_filtered[country_col].value_counts().head(25).to_string())

Top 25 countries in filtered sources ('pub_country'):

pub_country
ESP    35
IND    23
USA    16
KOR    15
ARG    12
VNM    11
ITA    11
TUR    10
GBR    10
UKR     9
SRB     9
CAN     8
COL     7
HRV     6
BRA     6
IDN     6
TWN     5
RUS     5
JPN     5
FRA     5
DEU     5
MEX     5
GRC     5
AZE     4
XKX     4


In [20]:
# Top 30 sources by content per week
if cpw_field and cpw_field in df_filtered.columns:
    top_cols = [c for c in ["name", "domain", cpw_field, "pub_country",
                            "national_collections"]
                if c in df_filtered.columns]
    print(f"Top 30 sources by {cpw_field}:\n")
    display(df_filtered[top_cols].head(30))

Top 30 sources by stories_per_week:



,name,stories_per_week,pub_country,national_collections
5583,bhaskar.com,45792.0,IND,India - National
5122,business24.ro,27348.0,None,Romania - National
11615,news.de,26208.0,DEU,Germany - National (Media Impact Monitor Test)...
5584,amarujala.com,23140.0,IND,India - National
5982,fnnews.com,18852.0,KOR,"Korea, Republic of - National"
2871,infobae.com,15974.0,ARG,Argentina - National | Colombia - National
5585,indiatimes.com,14455.0,IND,India - National
5586,indiatimes.com,10213.0,IND,India - National
376,prtimes.jp,9566.0,None,Japan - National
2072,bbc.com,9037.0,GBR,Afghanistan - National | Bangladesh - National...


## 8 — Cleanup

In [21]:
if Path(CHECKPOINT_FILE).exists():
    Path(CHECKPOINT_FILE).unlink()
    print("Checkpoint removed.")

print("\n🎉 Output files:")
for f in sorted(Path(".").glob("national_*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:,.0f} KB)")

Checkpoint removed.

🎉 Output files:
  national_collections.csv  (33 KB)
  national_sources_all_20260424.csv  (3,863 KB)
  national_sources_cpw_gt_1000_20260424.csv  (118 KB)
  national_sources_cpw_gt_300_20260424.csv  (364 KB)
